In [11]:
import torch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

batch_size, seq_len, emb_dim = 2, 5, 32
head_dim = 8

In [7]:
class SelfAttn(torch.nn.Module):
  def __init__(self, emb_dim=32, num_heads=4):
    super().__init__()
    self.head_dim = emb_dim // num_heads
    self.num_heads = num_heads

    self.attn = torch.nn.Linear(emb_dim, emb_dim * 3)
    self.output_proj = torch.nn.Linear(emb_dim, emb_dim)

  # x: (batch_size, seq_len, emb_dim)
  def forward(self, x):
    batch_size, seq_len, emb_dim = x.size(0), x.size(1), x.size(2)
    
    x = self.attn(x)
    q, k, v = torch.split(x, emb_dim, dim=-1)
    
    q = q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    k = k.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    v = v.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    
    print('q after reshape', q.size())
    
    attn_weights = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
    print('attn_weights', attn_weights.size())
    
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool).tril(diagonal=0)
    attn_weights = torch.where(mask == True, attn_weights, float('-inf'))
    
    attn_weights = torch.nn.functional.softmax(attn_weights, dim=-1)
    
    attn_result = attn_weights @ v
    
    attn_result = attn_result.permute(0, 2, 1, 3).reshape(batch_size, seq_len, emb_dim)
    
    return self.output_proj(attn_result)
  

x = torch.randn(batch_size, seq_len, emb_dim).to(device)
SelfAttn(emb_dim=emb_dim).to(device)(x).size()

q after reshape torch.Size([2, 4, 5, 8])
attn_weights torch.Size([2, 4, 5, 5])


torch.Size([2, 5, 32])

torch.Size([2, 5])